In [1]:
with open("/kaggle/input/datasets/ashishpandey2062/next-word-predictor-text-generator-dataset/next_word_predictor.txt", "r", encoding="utf-8") as file:
    text = file.read()


In [2]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for file in files:
        print("   ", file)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/ashishpandey2062
/kaggle/input/datasets/ashishpandey2062/next-word-predictor-text-generator-dataset
    next_word_predictor.txt


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [4]:
token = Tokenizer()
token.fit_on_texts([text])

In [5]:
len(token.word_index)
## total vocabe is 8930
# and we asing the int value to all words

4993

In [6]:
input_sequence=[]
# hear we split the all the text into sentance wise 
for sentance in text.split('.'):
    # we pass it in a list because we can add or convert more words into int using word index
    token_sentance = token.texts_to_sequences([sentance])[0]
    # we start it from 1 becaue we use it in y also 
    for words in range(1,len(token_sentance)):
        input_sequence.append(token_sentance[:words+1])

        
        




In [7]:
print(f'length of data row {len(input_sequence)} ')
input_sequence[:10]


length of data row 25878 


[[1, 155],
 [1, 155, 21],
 [1, 155, 21, 2368],
 [1, 155, 21, 2368, 1549],
 [1, 155, 21, 2368, 1549, 8],
 [1, 155, 21, 2368, 1549, 8, 1],
 [1, 155, 21, 2368, 1549, 8, 1, 422],
 [1, 155, 21, 2368, 1549, 8, 1, 422, 692],
 [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215],
 [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2]]

befor slicing we need to add padding to it

In [8]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [9]:
pad_sequence = pad_sequences(input_sequence,padding='pre')
print(pad_sequence)

[[   0    0    0 ...    0    1  155]
 [   0    0    0 ...    1  155   21]
 [   0    0    0 ...  155   21 2368]
 ...
 [   0    0    0 ... 2331  290   19]
 [   0    0    0 ...  290   19   54]
 [   0    0    0 ...   19   54 1535]]


In [10]:
y = pad_sequence[:,-1]
x = pad_sequence[:,:-1]

In [11]:
print(y)
x

[ 155   21 2368 ...   19   54 1535]


array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  155],
       [   0,    0,    0, ...,    1,  155,   21],
       ...,
       [   0,    0,    0, ...,   64, 2331,  290],
       [   0,    0,    0, ..., 2331,  290,   19],
       [   0,    0,    0, ...,  290,   19,   54]], dtype=int32)

In [12]:
# y is the word form
# know we need to convert it into one hot encoding

In [13]:
from tensorflow.keras.utils import to_categorical

In [14]:
print(max(y))
print(min(y))
print(len(token.word_index))

4993
1
4993


In [15]:
# we add 1 because we need to add the 0 in the vocabe becaue we use padding in it
y = to_categorical(y,num_classes=len(token.word_index)+1) # len(token.word_index)+1


In [16]:
x.shape

(25878, 83)

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Embedding , LSTM

In [18]:
model = Sequential()
model.add(Embedding(input_dim=4994 , output_dim=250,input_length=83 ))
model.add(LSTM(200,return_sequences=True))
model.add(LSTM(200,return_sequences=False))
model.add(Dense(4994,activation='softmax'))
model.build(input_shape=([None,83]))
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1784731019.435521      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784731019.438580      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 83, 250)        │     1,248,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 83, 200)        │       360,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 200)            │       320,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4994)           │     1,003,794 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,933,894 (11.19 MB)

 Trainable params: 2,933,894 (11.19 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
#model.fit(x,y,epochs=20,validation_split=0.2)

In [20]:
print(x.shape)
print(y.shape)
print(model.output_shape)

(25878, 83)
(25878, 4994)
(None, 4994)


In [21]:
import numpy as np 


In [22]:
inputs = 'deep' 
for i in range(5):
    token_text=token.texts_to_sequences([inputs])[0]
    padding_text = pad_sequences([token_text],maxlen =104 ,padding ='pre')
    pred = model.predict(padding_text)
    pred_index = np.argmax(pred)
    for word , index in token.word_index.items():
        if index==pred_index:

            inputs = inputs + ' ' + word
            print(inputs)
    



1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
deep vulnerability
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
deep vulnerability vulnerability
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
deep vulnerability vulnerability vulnerability
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
deep vulnerability vulnerability vulnerability vulnerability
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
deep vulnerability vulnerability vulnerability vulnerability vulnerability


## hypertuning for best parameters

In [23]:
import kerastuner as kt
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dropout
import keras_tuner

/tmp/ipykernel_58/3858820843.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [ ]:
# build model
def build_model(hp):
    model = Sequential()
    model.add(Embedding(input_dim=4994 , output_dim=hp.Choice('embedding_output',[100,150,200,300,250])))
    model.add(LSTM(units=hp.Int('lstm1',128,583,step=64),return_sequences=True,dropout=hp.Float("dropout1", 0.0, 0.5, step=0.1),recurrent_dropout=hp.Float("recurrent_dropout1", 0.0, 0.5, step=0.1)))
    model.add(LSTM(units=hp.Int('lstm2',64,583,step=64),return_sequences=False,dropout=hp.Float("dropout2", 0.0, 0.5, step=0.1)))
    model.add(Dense(4994,activation='softmax'))
    model.build(input_shape=([None,83]))
    model.compile(loss='categorical_crossentropy',optimizer=Adam(learning_rate=hp.Choice('lr',[0.01,0.002])),metrics=['accuracy'])
    model.summary()
    return model

In [71]:
# max_trials mean random 5 times parameters pick ker ke train kerna
tuner =kt.RandomSearch(
    build_model,
    objective = 'val_accuracy',
    max_trials = 5
)

Reloading Tuner from ./untitled_project/tuner0.json


In [72]:
print(x.shape)
print(y.shape)
print(len(token.index_word))

(25878, 83)
(25878, 4994)
4993


In [73]:
tuner.search(x,y,epochs=5,validation_split=.2)

In [75]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)


{'embedding_output': 250, 'lstm1': 192, 'dropout1': 0.2, 'recurrent_dropout1': 0.30000000000000004, 'lstm2': 512, 'dropout2': 0.4, 'lr': 0.005}


In [63]:
base_model = tuner.hypermodel.build(best_hp)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 83, 250)        │     1,248,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 83, 192)        │       340,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 512)            │     1,443,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4994)           │     2,561,922 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,594,486 (21.34 MB)

 Trainable params: 5,594,486 (21.34 MB)

 Non-trainable params: 0 (0.00 B)

In [81]:
from tensorflow.keras.callbacks import EarlyStopping

In [82]:
base_model.fit(x,y,epochs=50,batch_size =300 ,callbacks=EarlyStopping(monitor='val_loss',
                                                    patience=5,
                                                   mode='auto',# decresing 
                                                   min_delta = 0,
                                                   restore_best_weights=True))

Epoch 1/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 262ms/step - accuracy: 0.1208 - loss: 5.5721
Epoch 2/50


/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 264ms/step - accuracy: 0.1269 - loss: 5.3854
Epoch 3/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 261ms/step - accuracy: 0.1300 - loss: 5.2256
Epoch 4/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 261ms/step - accuracy: 0.1321 - loss: 5.0512
Epoch 5/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 262ms/step - accuracy: 0.1359 - loss: 4.8894
Epoch 6/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 264ms/step - accuracy: 0.1453 - loss: 4.6979
Epoch 7/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 260ms/step - accuracy: 0.1535 - loss: 4.5207
Epoch 8/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 260ms/step - accuracy: 0.1655 - loss: 4.3490
Epoch 9/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 262ms/step - accuracy: 0.1803 - loss: 4.1689
Epoch 10/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 260ms/step - accuracy: 0.2008 - loss: 3.9844
Epoch 11/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 261ms/step - accuracy: 0.2252 - loss: 3.8047
Epoch 12/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 259ms/step - accuracy: 0.2467 - loss: 3.6409
Epoch 13/50
87/87 ━━━━━━━━━━━━━━━━━━━━ 23s 261ms/st

In [78]:
print(base_model.optimizer.learning_rate.numpy())

0.005


In [90]:
inputs = 'Quantum computers' 
for i in range(40):
    token_text=token.texts_to_sequences([inputs])[0]
    padding_text = pad_sequences([token_text],maxlen =104 ,padding ='pre')
    pred = base_model.predict(padding_text)
    pred_index = np.argmax(pred)
    for word , index in token.word_index.items():
        if index==pred_index:

            inputs = inputs + ' ' + word
            print(inputs)
    



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Quantum computers harness
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
Quantum computers harness the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
Quantum computers harness the principles
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Quantum computers harness the principles of
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
Quantum computers harness the principles of quantum
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
Quantum computers harness the principles of quantum mechanics
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
Quantum computers harness the principles of quantum mechanics to
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
Quantum computers harness the principles of quantum mechanics to perform
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
Quantum computers harness the principles of quantum mechanics to perform calculations
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
Quantum computers harness the principles of quantum mechanics to perform calculations far
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
Qu

In [101]:
base_model.save("lstm_next_word.keras")

In [103]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(token, f)

In [105]:
max_len = x.shape[1]
max_len

83

In [106]:
with open("max_len(time_stamp).","wb") as f:
    pickle.dump(max_len,f)